# 🫀 실험 4b — 기권 규칙을 **예측 클래스별로** 나눈다

**MedKOS / `notebooks/exp4b_class_conditional_abstain.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 실험4가 남긴 것과, 그걸 그대로 쓸 수 없는 이유

실험4 CELL 6.5에서 오답 탐지 AUC를 클래스별로 쪼개니 이렇게 나왔습니다:

| | N | S | V |
|---|---|---|---|
| 품질(추정 SNR) | 0.419 | 0.610 | **0.752** |
| 확률(최대 softmax) | **0.878** | **0.184** | 0.789 |

**확률의 S 내부 AUC 0.184는 무정보(0.5)가 아니라 역전입니다.** 진짜 S 비트 안에서
confidence가 높을수록 틀립니다. 기권에 쓰면 S에서 **맞힌 것을 버리고 틀린 것을 남깁니다.**

> **그런데 이 표는 그대로 배포할 수 없습니다.**
> `0.184`는 "**진짜** S 비트들 중에서"입니다. 추론 시점에 어느 게 진짜 S인지 알면
> 애초에 분류기가 필요 없습니다.

배포 가능한 조건은 **예측 클래스**뿐입니다. 그리고 이건 참 클래스와 같은 집합이 아닙니다 —
S의 F1이 낮다는 건 **진짜 S의 상당수가 N으로 예측된다**는 뜻이고, 그래서 "예측 N" 바구니
안에 진짜 S가 섞여 있습니다.

**그러므로 참 클래스에서 본 역전이 예측 클래스로 넘어갈지는 별개 문제입니다.**
희석돼서 사라질 수도 있습니다. **이 실험은 진짜로 실패할 수 있습니다.**

## 무엇을 비교하는가 — 교란을 제거한 설계

순진하게 "클래스조건부 규칙 vs 전역 확률"을 비교하면 **두 가지가 섞입니다**:
① 클래스로 조건을 걸어서 좋아진 것, ② 품질 특징을 추가해서 좋아진 것.

그래서 **특징 집합을 고정한 채 조건화만 바꿉니다**:

| 규칙 | 입력 | 조건화 | 역할 |
|---|---|---|---|
| **전역 로지스틱** | `[conf, snr_hat]` | 없음 | **주 비교 대상** |
| **클래스조건부 로지스틱** | `[conf, snr_hat]` | **예측** 클래스별 | **주가설** |
| 오라클 | `[conf, snr_hat]` | **참** 클래스별 | 상한(배포 불가) |
| 전역 확률 | `conf` | 없음 | 실험4의 승자(실무 기준선) |
| 전역 품질 | `snr_hat` | 없음 | 실험4의 사전등록 기준선 |
| 무작위 | — | — | **필수 대조군** |

로지스틱을 쓰면 **부호를 손으로 고를 필요가 없습니다.** N에서는 conf 계수가 음수
(확신↑ → 오답↓), S에서는 양수(역전)로 **데이터가 알아서 정합니다.**

## 사전등록

| # | 예측 | 판정 |
|---|---|---|
| **G0 (관문)** | 오라클 − 전역로지스틱 ≥ **0.01** 이고 유의 | 못 넘으면 **여기서 중단** |
| **H1 (주가설)** | 클래스조건부 − 전역로지스틱 > 0 유의 | 조건화 자체의 이득 |
| **H2** | 클래스조건부 − 전역확률 > 0 유의 | 실무 기준선 대비 이득 |
| **H3** | 예측 S 안에서 conf의 오답탐지 AUC < 0.45 | 참 클래스의 역전이 예측 클래스로 전이되는가 |

**G0가 관문인 이유**: `ailab-2026-0018` §4-bis에서 환자 선별 아이디어를 닫았던 규율입니다.
**클래스 조건화로 얻을 게 있는지부터 재고, 없으면 시간을 쓰지 않습니다.**

> ⚠️ **오라클은 "참고 상한"이지 수학적 상한이 아닙니다.** 아래에서 기권 점수를
> **클래스 내 백분위**로 정규화하는데, 참 클래스로 묶는 것과 예측 클래스로 묶는 것은
> **서로 포함관계가 아닌 다른 분할**입니다. macro-F1은 참 라벨로 채점되므로 참 클래스로
> 묶으면 클래스별 잔존 비율이 직접 통제되고, 예측 클래스로 묶으면 그렇지 않습니다.
> 그래서 배포 가능 버전이 오라클을 **넘는 것도 가능**합니다(픽스처에서 109% 관측).
> G0는 "조건화에 여지가 있는가"를 재는 것이지 천장을 긋는 게 아닙니다.

## 과적합 방지 — 이게 없으면 전부 무의미하다

규칙(클래스별 계수)을 **테스트셋에서 고르면 그건 검정이 아니라 커브 피팅**입니다.
DS2 환자를 **환자 단위로 반 갈라** 한쪽에서 규칙을 적합하고 다른 쪽에서만 채점합니다.

```
DS2 22명  →  A조 11명 (규칙 적합)  |  B조 11명 (검정 — 여기 숫자만 보고한다)
```

부트스트랩도 **환자 클러스터 단위**로 합니다(실험4는 비트 단위였음 — 비트는 환자 안에서
독립이 아니라 CI가 과소추정됩니다). 대신 B조가 11명뿐이라 **CI가 넓게 나올 것**이고,
그건 정직한 대가입니다. 비교용으로 비트 단위 CI도 같이 찍습니다.

## 비용
**재학습 없음.** 실험4가 Drive에 남긴 분류기·품질추정기 가중치와 비트 캐시를 재사용합니다.


In [ ]:
# CELL 1 — 설정 + 실험4 산출물 찾기
!pip -q install wfdb

import os, sys, json, time, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★ 실험4와 완전히 같아야 하는 상수들 — 하나라도 다르면 혼합 테스트셋이 재현되지 않는다
FS, W_PRE, W_POST = 360, 144, 216
SNRS = [24, 18, 12, 6, 0, -6]
CLASSES = ["N", "S", "V"]
SEED0 = 20260731          # 실험4의 시드(20260801 아님 — 재현에 필요)
COV_MAIN = 0.8
COVS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
MIN_GAIN = 0.01           # G0 관문: 오라클이 이만큼은 넘겨야 진행할 가치가 있다
SPLIT_SEED = 4242

CONFIG = dict(exp="exp4b_class_conditional_abstain", quest="ailab-2026-0015",
              parent_exp="exp4_quality",
              hypothesis="기권 규칙을 예측 클래스별로 나누면 전역 단일 규칙을 이기는가",
              gate="G0 오라클 − 전역로지스틱 ≥ 0.01 유의 (못 넘으면 중단)",
              primary="H1 클래스조건부 − 전역로지스틱 > 0 @cov 0.8",
              features=["max_softmax", "snr_hat"], coverage=COV_MAIN,
              rule_fit="DS2 환자 절반(A조) · 검정은 나머지 절반(B조)",
              bootstrap="환자 클러스터(주) + 비트(참고)", seed0=SEED0,
              split_seed=SPLIT_SEED, boot=2000)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as _e:
    print("한글 폰트 설정 생략:", _e)

run = MedKOSRun("exp4b_cond_abstain", CONFIG, project=PROJECT)

# ── 실험4 실행 폴더를 registry.jsonl에서 찾는다(경로를 손으로 적지 않는다)
REG = os.path.join(PROJECT, "registry.jsonl")
prev = None
if os.path.exists(REG):
    for line in open(REG):
        try:
            r = json.loads(line)
        except Exception:
            continue
        if r.get("exp_id") == "exp4_quality" and os.path.isdir(r.get("dir", "")):
            prev = r                      # 마지막(=가장 최근) 것을 쓴다
if prev is None:
    raise RuntimeError("registry.jsonl에서 exp4_quality 실행을 못 찾았습니다. "
                       "실험4를 먼저 돌리세요.")
PREV_DIR = prev["dir"]
run.log(f"실험4 산출물: {PREV_DIR}")
run.log(f"  요약: {prev.get('summary','')}")
for arm in ("classifier", "quality"):
    p = os.path.join(PREV_DIR, "arms", arm, "weights.keras")
    run.log(f"  {'✅' if os.path.exists(p) else '❌'} {arm}: {p}")


In [ ]:
# CELL 2 — 비트 캐시 + 노이즈 + 저장된 모델 로드 (재학습 없음)
import wfdb, tensorflow as tf

DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]

CACHE = run.data(f"mitdb_beats_{FS}hz.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"비트 캐시가 없습니다: {CACHE} — 실험4 CELL 3을 먼저 돌리세요")
z = np.load(CACHE); Xc, Yc, Pc = z["X"], z["y"], z["pid"]
run.log(f"비트 {len(Yc):,} · 환자 {len(np.unique(Pc))} · 클래스 {np.bincount(Yc).tolist()}")

NOISE = {}
for nm in ("bw", "em", "ma"):
    r_ = wfdb.rdrecord(nm, pn_dir="nstdb")
    NOISE[nm] = r_.p_signal[:, 0].astype("float32")
run.log(f"nstdb 노이즈 {list(NOISE)} 확보")

def add_noise(X, snr_db, rng):
    """실험4와 **완전히 동일한 구현**이어야 한다 — 난수 소비 순서까지 같아야 재현된다."""
    out = np.empty_like(X)
    keys = list(NOISE)
    for i in range(len(X)):
        nz = NOISE[keys[rng.randint(len(keys))]]
        st = rng.randint(0, len(nz) - X.shape[1])
        n = nz[st:st + X.shape[1]]
        n = n - n.mean()
        ps, pn = np.mean(X[i] ** 2), np.mean(n ** 2) + 1e-12
        out[i] = X[i] + n * np.sqrt(ps / (pn * 10 ** (snr_db / 10)))
    return out

CLF = tf.keras.models.load_model(os.path.join(PREV_DIR, "arms", "classifier", "weights.keras"))
QUAL = tf.keras.models.load_model(os.path.join(PREV_DIR, "arms", "quality", "weights.keras"))
run.log("분류기·품질추정기 로드 완료 (재학습 없음)")


### CELL 3 — 실험4의 혼합 테스트셋을 **정확히** 재현했는지 검증

같은 난수 시퀀스를 소비해야 같은 테스트셋이 나옵니다. 재현이 안 되면 아래 모든 비교가
실험4와 다른 데이터 위에서 도는 것이라 **여기서 멈춰야** 합니다.

실험4가 출력한 값을 그대로 박아두고 대조합니다 — 통과 못 하면 예외를 던집니다.


In [ ]:
# CELL 3 — 혼합 테스트셋 재현 + 대조 검증
from sklearn.metrics import f1_score

te_idx = np.where(np.isin(Pc, DS2))[0]
rng3 = np.random.RandomState(SEED0 + 13)          # ★ 실험4 CELL 6과 동일
mix_snr = rng3.choice(SNRS, size=len(te_idx))
# ★ Xc[te_idx]를 루프 **밖으로** 뺀다. 컴프리헨션 안에 두면 fancy indexing이
#   매 반복마다 71MB 배열을 새로 복사한다(49,293회 = 3.5TB 메모리 트래픽).
#   실측 20.5분 → 2초. 난수 소비 순서는 그대로라 결과는 비트 단위로 동일하다.
Xte = Xc[te_idx]
Xmix = np.stack([add_noise(Xte[i:i+1], mix_snr[i], rng3)[0]
                 for i in range(len(te_idx))])[..., None]
ymix = Yc[te_idx]
pmix = Pc[te_idx]                                  # 환자 id — 실험4엔 없던 축
pr = CLF.predict(Xmix, batch_size=1024, verbose=0)
snr_hat = QUAL.predict(Xmix, batch_size=1024, verbose=0).ravel()
conf = pr.max(1)
pred = pr.argmax(1)
wrong = (pred != ymix)

dist = {int(s): int((mix_snr == s).sum()) for s in SNRS}
run.log(f"혼합 테스트셋 {len(ymix):,}비트 · SNR 분포 {dist}")

# ── 실험4가 실제로 출력한 값(하드코딩) — 재현 검증용
EXPECT_N = 49293
EXPECT_DIST = {24: 8127, 18: 8159, 12: 8309, 6: 8183, 0: 8193, -6: 8322}
EXPECT_CONF_CURVE = [0.564, 0.595, 0.615, 0.628, 0.640, 0.650]

def curve(score):
    ys = []
    for cov in COVS:
        k = int(len(ymix) * cov)
        keep = np.argsort(-score)[:k]
        ys.append(float(f1_score(ymix[keep], pred[keep], average="macro", zero_division=0)))
    return ys

got_curve = curve(conf)
run.log(f"전역 확률 곡선 재현 {[round(v,3) for v in got_curve]}")
run.log(f"           실험4 값 {EXPECT_CONF_CURVE}")

ok_n = (len(ymix) == EXPECT_N)
ok_d = (dist == EXPECT_DIST)
ok_c = all(abs(a - b) < 0.002 for a, b in zip(got_curve, EXPECT_CONF_CURVE))
run.log(f"  비트수 {'✅' if ok_n else '❌'} · SNR분포 {'✅' if ok_d else '❌'} · "
        f"곡선 {'✅' if ok_c else '❌'}")
if not (ok_n and ok_d and ok_c):
    raise RuntimeError("실험4의 테스트셋을 재현하지 못했습니다. 상수(FS/SNRS/SEED0)나 "
                       "add_noise 구현이 실험4와 다릅니다 — 비교가 무의미하므로 중단합니다.")
run.log("✅ 재현 확인 — 실험4와 동일한 데이터 위에서 비교합니다")
run.save_json("reproduction_check", {"n": int(len(ymix)), "snr_dist": dist,
                                     "conf_curve": got_curve, "passed": True})


In [ ]:
# CELL 4 — 환자 단위 A조/B조 분할 + 예측 클래스별 AUC (6.5의 배포 가능 버전)
pats = np.array(sorted(np.unique(pmix)))
rs = np.random.RandomState(SPLIT_SEED)
perm = rs.permutation(len(pats))
A_PATS, B_PATS = pats[perm[:len(pats)//2]], pats[perm[len(pats)//2:]]
A = np.isin(pmix, A_PATS); B = np.isin(pmix, B_PATS)
run.log(f"A조(규칙 적합) 환자 {len(A_PATS)}명 · 비트 {A.sum():,}")
run.log(f"B조(검정)      환자 {len(B_PATS)}명 · 비트 {B.sum():,}")
for nm, m in (("A", A), ("B", B)):
    run.log(f"  {nm}조 참클래스 {np.bincount(ymix[m], minlength=3).tolist()} · "
            f"예측클래스 {np.bincount(pred[m], minlength=3).tolist()} · "
            f"오답률 {wrong[m].mean():.3f}")
thin = [CLASSES[c] for c in range(3)
        if min((ymix[A] == c).sum(), (ymix[B] == c).sum()) < 50]
if thin:
    run.log(f"  ⚠️ 표본이 얇은 클래스: {thin} — 해당 클래스 결과는 해석 주의")

def auc(score, mask):
    """낮을수록 오답인가 → 오답 탐지 AUC (Mann-Whitney U)."""
    y, x = wrong[mask], -score[mask]
    if y.sum() < 10 or (~y).sum() < 10:
        return None
    r = np.argsort(np.argsort(x)) + 1.0
    n1, n0 = int(y.sum()), int((~y).sum())
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

run.log("\n" + "=" * 88)
run.log("【H3】 참 클래스의 역전이 **예측** 클래스로 전이되는가 (B조에서만)")
run.log("=" * 88)
run.log(f"{'조건':<16}{'n':>8}{'오답률':>9}{'conf AUC':>11}{'snr AUC':>11}")
run.log("-" * 88)
cond_auc = {}
for kind, lab in (("true", "참"), ("pred", "예측")):
    base = ymix if kind == "true" else pred
    for c in range(3):
        m = B & (base == c)
        a_c, a_s = auc(conf, m), auc(snr_hat, m)
        cond_auc[f"{kind}_{CLASSES[c]}"] = {"n": int(m.sum()),
                                            "err": float(wrong[m].mean()) if m.sum() else None,
                                            "conf_auc": a_c, "snr_auc": a_s}
        f = lambda v: f"{v:>11.3f}" if v is not None else f"{'n/a':>11}"
        run.log(f"{lab+' '+CLASSES[c]:<16}{int(m.sum()):>8,}"
                f"{(wrong[m].mean() if m.sum() else 0):>9.3f}{f(a_c)}{f(a_s)}")
run.log("-" * 88)

ps = cond_auc["pred_S"]["conf_auc"]
ts = cond_auc["true_S"]["conf_auc"]
h3 = bool(ps is not None and ps < 0.45)
run.log(f"\nH3 — 예측 S 안에서 conf AUC = "
        f"{f'{ps:.3f}' if ps is not None else 'n/a'} "
        f"(참 S에서는 {f'{ts:.3f}' if ts is not None else 'n/a'})")
run.log(f"   → {'✅ 역전이 전이됨' if h3 else '❌ 전이 안 됨(희석되었거나 표본 부족)'}")
run.log("   전이가 안 되어도 H1은 성립할 수 있다 — 조건화의 이득이 S 역전에서만 오는 건 아니다.")
run.save_json("conditional_auc", {"per_condition": cond_auc, "H3": h3})


In [ ]:
# CELL 5 — 【G0 관문】 오라클 상한을 먼저 잰다
from sklearn.linear_model import LogisticRegression

FEAT = np.c_[conf, snr_hat].astype("float64")
mu, sd = FEAT[A].mean(0), FEAT[A].std(0) + 1e-9
Z = (FEAT - mu) / sd

def fit_lr(mask):
    """오답 확률 P(wrong | conf, snr_hat). 표본이 없거나 한쪽 라벨뿐이면 None."""
    y = wrong[mask]
    if mask.sum() < 30 or y.sum() < 5 or (~y).sum() < 5:
        return None
    return LogisticRegression(max_iter=1000, C=1.0).fit(Z[mask], y)

def score_global():
    m = fit_lr(A)
    if m is None:
        raise RuntimeError("전역 로지스틱을 적합할 수 없습니다 — A조 표본 확인")
    return m.predict_proba(Z)[:, 1], m

def score_conditional(basis):
    """basis: 'pred'(배포 가능) 또는 'true'(오라클, 배포 불가)."""
    gm = fit_lr(A)
    out = gm.predict_proba(Z)[:, 1].copy()      # 기본값 = 전역 모델
    key = pred if basis == "pred" else ymix
    used = {}
    for c in range(3):
        mc = fit_lr(A & (key == c))
        if mc is None:
            used[CLASSES[c]] = "전역 대체(표본 부족)"
            continue
        sel = (key == c)
        out[sel] = mc.predict_proba(Z[sel])[:, 1]
        used[CLASSES[c]] = {"coef_conf": float(mc.coef_[0][0]),
                            "coef_snr": float(mc.coef_[0][1])}
    return out, used

perr_global, GM = score_global()
perr_pred, used_pred = score_conditional("pred")
perr_true, used_true = score_conditional("true")

def within_class_rank(perr, key):
    """★ 클래스별 로지스틱은 절편이 다르다. P(오답)을 그대로 전역 랭킹에 넣으면
       기저 오답률이 높은 희소 클래스가 상위에 통째로 몰려 **그 클래스가 소멸**하고
       macro-F1이 1/K로 붕괴한다(픽스처에서 실제로 발생).
       클래스 안에서 백분위로 바꾸면 각 클래스가 비슷한 비율로 남으면서,
       클래스 내부에서는 위험한 것부터 버린다."""
    out = np.zeros_like(perr)
    for c in range(3):
        sel = np.where(key == c)[0]
        if len(sel) == 0:
            continue
        r = np.argsort(np.argsort(perr[sel])).astype("float64")
        out[sel] = r / max(len(sel) - 1, 1)
    return out

perr_pred_rank = within_class_rank(perr_pred, pred)
perr_true_rank = within_class_rank(perr_true, ymix)

run.log("적합된 계수 (표준화 특징 · 부호 +면 '값이 클수록 오답')")
run.log(f"  전역        conf {GM.coef_[0][0]:+.3f} · snr {GM.coef_[0][1]:+.3f}")
for c in CLASSES:
    u = used_pred[c]
    run.log(f"  예측={c:<3s}   " + (u if isinstance(u, str) else
            f"conf {u['coef_conf']:+.3f} · snr {u['coef_snr']:+.3f}"))
run.log("  ※ conf 계수가 **음수**면 '확신할수록 맞다'(정상), **양수**면 역전이다.")
run.log("     손으로 부호를 고르지 않았다 — 데이터가 정했다.")

# 기권 점수는 '오답 확률'이므로 낮을수록 남긴다 → 부호를 뒤집어 curve()에 넣는다
def curveB(score_keep):
    ys = []
    idxB = np.where(B)[0]
    for cov in COVS:
        k = int(len(idxB) * cov)
        keep = idxB[np.argsort(-score_keep[idxB])[:k]]
        ys.append(float(f1_score(ymix[keep], pred[keep], average="macro", zero_division=0)))
    return ys

RULES = {
    "무작위":              np.random.RandomState(SEED0 + 7).rand(len(ymix)),
    "전역 품질":           snr_hat,
    "전역 확률":           conf,
    "전역 로지스틱":       -perr_global,
    "클래스조건부(예측)":   -perr_pred_rank,     # ★ 주가설 = 클래스내 순위 버전
    "오라클(참클래스)":     -perr_true_rank,
    "[참고] 조건부-원확률": -perr_pred,          # 절편까지 쓰는 버전(붕괴 위험)
}
curves = {k: curveB(v) for k, v in RULES.items()}

# ── 붕괴 가드: keep 집합에서 클래스가 통째로 사라지면 macro-F1은 무의미해진다
idxB_ = np.where(B)[0]
collapse = {}
for k, v in RULES.items():
    kk = int(len(idxB_) * COV_MAIN)
    keep = idxB_[np.argsort(-v[idxB_])[:kk]]
    cnt = np.bincount(ymix[keep], minlength=3)
    gone = [CLASSES[i] for i in range(3) if cnt[i] == 0]
    collapse[k] = {"kept_per_class": cnt.tolist(), "vanished": gone}
    if gone:
        run.log(f"  ⛔ {k}: 커버리지 {COV_MAIN:.0%}에서 클래스 {gone} 소멸 "
                f"→ macro-F1 해석 불가")
run.log("  (클래스가 소멸하면 macro-F1이 1/K로 붕괴한다 — 성능이 아니라 붕괴다)")
run.log("\nB조 커버리지 곡선")
run.log(f"  {'규칙':<20}" + "".join(f"{c:>8.0%}" for c in COVS))
for k, v in curves.items():
    run.log(f"  {k:<20}" + "".join(f"{x:>8.3f}" for x in v))

i80 = COVS.index(COV_MAIN)
oracle_gain = curves["오라클(참클래스)"][i80] - curves["전역 로지스틱"][i80]
run.log(f"\n【G0 관문】 오라클 − 전역로지스틱 @{COV_MAIN:.0%} = {oracle_gain:+.4f} "
        f"(기준 ≥ {MIN_GAIN})")
G0_point = bool(oracle_gain >= MIN_GAIN)
run.log(f"  점추정 {'통과' if G0_point else '미달'} — 유의성은 CELL 6에서 확인")
if not G0_point:
    run.log("  ⚠️ 상한이 이미 기준 아래다. 배포 가능한 버전은 상한을 넘을 수 없으므로")
    run.log("     H1이 유의하게 나오더라도 실용적 가치는 없다고 보고할 것.")


In [ ]:
# CELL 6 — 부트스트랩: 환자 클러스터(주) + 비트(참고)
def boot(score_a, score_b, cov=COV_MAIN, mode="patient", B_=CONFIG["boot"], seed=SEED0):
    """B조에서 두 규칙의 macro-F1 차이. mode='patient'면 환자를 재표본(클러스터)."""
    rs = np.random.RandomState(seed)
    idxB = np.where(B)[0]
    by_pat = {p: idxB[pmix[idxB] == p] for p in B_PATS}
    out = []
    for _ in range(B_):
        if mode == "patient":
            pick = rs.choice(B_PATS, len(B_PATS), replace=True)
            ii = np.concatenate([by_pat[p] for p in pick])
        else:
            ii = idxB[rs.randint(0, len(idxB), len(idxB))]
        k = int(len(ii) * cov)
        d = []
        for s in (score_a, score_b):
            keep = ii[np.argsort(-s[ii])[:k]]
            d.append(f1_score(ymix[keep], pred[keep], average="macro", zero_division=0))
        out.append(d[0] - d[1])
    out = np.array(out)
    return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

COMPARE = [
    ("G0 오라클 − 전역로지스틱",      "오라클(참클래스)",   "전역 로지스틱"),
    ("H1 클래스조건부 − 전역로지스틱", "클래스조건부(예측)", "전역 로지스틱"),
    ("H2 클래스조건부 − 전역확률",     "클래스조건부(예측)", "전역 확률"),
    ("   클래스조건부 − 무작위",       "클래스조건부(예측)", "무작위"),
    ("   전역확률 − 무작위",           "전역 확률",         "무작위"),
    ("   [참고] 조건부원확률 − 조건부순위", "[참고] 조건부-원확률", "클래스조건부(예측)"),
]
run.log("=" * 96)
run.log(f"{'비교':<32}{'Δ(환자CI)':>12}{'95% CI':>24}{'':>4}{'Δ(비트CI)':>12}{'비트 95% CI':>22}")
run.log("=" * 96)
res = {}
for label, a, b in COMPARE:
    d, l, u = boot(RULES[a], RULES[b], mode="patient")
    d2, l2, u2 = boot(RULES[a], RULES[b], mode="beat")
    sig = bool(l > 0 or u < 0)
    res[label.strip()] = {"delta": d, "ci_patient": [l, u], "significant_patient": sig,
                          "delta_beat": d2, "ci_beat": [l2, u2],
                          "significant_beat": bool(l2 > 0 or u2 < 0)}
    run.log(f"{label:<32}{d:>+12.4f}   [{l:+.4f}, {u:+.4f}] "
            f"{'★' if sig else ' '}{d2:>+12.4f}   [{l2:+.4f}, {u2:+.4f}]")
run.log("=" * 96)
run.log(f"  환자 CI가 주지표(B조 {len(B_PATS)}명 클러스터). 비트 CI는 실험4와 비교하려고 같이 찍는다 —")
run.log("  비트는 환자 안에서 독립이 아니므로 비트 CI는 구조적으로 좁게 나온다.")

g0 = res["G0 오라클 − 전역로지스틱"]
h1 = res["H1 클래스조건부 − 전역로지스틱"]
h2 = res["H2 클래스조건부 − 전역확률"]
G0_pass = bool(g0["significant_patient"] and g0["delta"] >= MIN_GAIN)

run.log("\n" + "=" * 96)
run.log("【판정】")
run.log("=" * 96)
run.log(f"  G0 관문: Δ={g0['delta']:+.4f} [{g0['ci_patient'][0]:+.4f}, "
        f"{g0['ci_patient'][1]:+.4f}] · 기준 ≥{MIN_GAIN} → {'통과' if G0_pass else '미달'}")
if not G0_pass:
    verdict = ("기각(관문) — 참 클래스를 알려줘도 조건화 이득이 기준 미만이다. "
               "클래스 조건화라는 축 자체에 여지가 없다는 뜻이므로 더 파지 않는다. "
               "abstain은 전역 규칙으로 두고, 기권의 1축은 확신도가 아니라 "
               "**관측가능성(유도 구성)** 으로 간다")
    run.log(f"  → **{verdict}**")
    run.log("     (실험10이 관측가능성 축을 정량화한다 — 그쪽이 더 강한 기권 근거다)")
else:
    run.log(f"  H1 주가설: Δ={h1['delta']:+.4f} [{h1['ci_patient'][0]:+.4f}, "
            f"{h1['ci_patient'][1]:+.4f}] {'유의 ★' if h1['significant_patient'] else '비유의'}")
    run.log(f"  H2 실무기준: Δ={h2['delta']:+.4f} [{h2['ci_patient'][0]:+.4f}, "
            f"{h2['ci_patient'][1]:+.4f}] {'유의 ★' if h2['significant_patient'] else '비유의'}")
    frac = h1["delta"] / (g0["delta"] + 1e-9)
    run.log(f"  배포 가능 버전이 오라클 대비 {frac*100:.0f}% (100% 초과 가능 — 오라클은")
    run.log("  참 클래스로 묶은 다른 분할이라 수학적 상한이 아니다)")
    if h1["significant_patient"] and h1["delta"] > 0 and h2["significant_patient"] and h2["delta"] > 0:
        verdict = (f"확증 — 예측 클래스 조건부 기권이 전역 규칙을 이긴다(오라클의 {frac*100:.0f}% 회수). "
                   "abstain 헤드를 클래스별로 분기한다")
    elif h1["significant_patient"] and h1["delta"] > 0:
        verdict = ("부분 확증 — 조건화 자체는 이득이나 실무 기준선(전역 확률)은 못 이겼다. "
                   "이득이 품질 특징이 아니라 조건화에서 왔다는 것까지만 확정")
    else:
        verdict = ("미결 — 오라클엔 여지가 있으나 예측 클래스로는 회수하지 못했다. "
                   "참→예측 전이에서 신호가 희석된다. 더 나은 조건화 변수(예: 상위2 확률 마진)가 필요")
    run.log(f"  → **{verdict}**")
run.log("=" * 96)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for k, v in curves.items():
    st = "--" if k.startswith("오라클") else ("-." if k == "무작위" else "o-")
    ax.plot([c * 100 for c in COVS], v, st, label=k, lw=2 if "조건부" in k else 1.3)
ax.set_xlabel("커버리지 (%)"); ax.set_ylabel("macro-F1 (B조)"); ax.invert_xaxis()
ax.set_title("실험4b — 기권 전략 (B조: 규칙 적합에 안 쓴 환자)")
ax.legend(fontsize=8); plt.tight_layout(); run.save_fig("abstain_conditional", fig); plt.show()

run.save_json("evaluation", {"curves": curves, "comparisons": res,
                             "collapse_check": collapse,
                             "coefficients": {"global": GM.coef_[0].tolist(),
                                              "per_pred_class": used_pred,
                                              "per_true_class": used_true},
                             "G0_pass": G0_pass, "verdict": verdict})

result = {"week": 1, "exp_id": "exp4b_cond_abstain", "quest": "ailab-2026-0015",
          "task": "예측 클래스 조건부 기권 규칙이 전역 단일 규칙을 이기는가",
          "split": "inter", "metric": "macro_f1_at_cov80_heldout_patients",
          "value": round(curves["클래스조건부(예측)"][i80], 4),
          "passed": bool(G0_pass and h1["significant_patient"] and h1["delta"] > 0),
          "date": time.strftime("%Y-%m-%d"),
          "n_patients_fit": len(A_PATS), "n_patients_test": len(B_PATS),
          "oracle_gain": g0["delta"], "G0_pass": G0_pass,
          "comparisons": res, "H3_inversion_transfers": h3, "verdict": verdict,
          "summary": (f"G0 오라클 {g0['delta']:+.4f} → {'통과' if G0_pass else '미달'} · "
                      f"H1 {h1['delta']:+.4f} [{h1['ci_patient'][0]:+.4f},"
                      f"{h1['ci_patient'][1]:+.4f}] · {verdict.split(' —')[0]}")}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp4b_class_conditional_abstain.ipynb \\
      --quest ailab-2026-0015 --step "exp4b-class-conditional-abstain" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")


---

## 결과 읽는 법

| G0 | H1 | 뜻 | 다음 |
|---|---|---|---|
| **미달** | — | 참 클래스를 알려줘도 조건화 이득이 없다 = **축 자체에 여지가 없다** | abstain은 전역 규칙 유지. 기권의 1축을 **관측가능성(실험10)** 으로 옮긴다 |
| 통과 | **유의 +** | 조건화가 실제로 작동. 오라클의 몇 %를 회수했는지가 실용 지표 | abstain 헤드를 예측 클래스별로 분기 |
| 통과 | 비유의 | 여지는 있는데 **참→예측 전이에서 희석**됨 | 조건화 변수를 바꾼다(상위2 확률 마진, 예측 엔트로피) |

## 이 실험이 지키는 규율

1. **오라클을 먼저 잰다.** 배포 가능한 버전이 상한을 넘을 수는 없다. 상한이 작으면
   거기서 끝낸다 — `ailab-2026-0018` §4-bis에서 환자 선별 아이디어를 닫았던 방식.
2. **규칙 적합과 검정을 환자 단위로 분리한다.** 테스트셋에서 규칙을 고르면 검정이 아니다.
3. **특징을 고정하고 조건화만 바꾼다.** 전역 로지스틱과 클래스조건부 로지스틱은 입력이
   같다 — 차이는 오직 조건화다. 전역 확률과 비교하면 '품질 추가'와 '조건화'가 섞인다.
4. **부호를 손으로 고르지 않는다.** 로지스틱 계수가 정한다. S에서 conf 계수가 양수로
   나오면 그게 역전의 증거고, 우리가 심어준 게 아니다.
5. **환자 클러스터 부트스트랩.** 비트는 환자 안에서 독립이 아니다. CI가 넓어지는 건
   정직한 대가다(실험4의 비트 CI는 구조적으로 좁았다).
6. **무작위 대조군.** 기권은 무엇으로 하든 남은 표본 성능을 올린다.

## 한계

- **B조 11명.** 환자 클러스터 부트스트랩의 CI가 넓다. 비유의가 나와도 "효과 없음"이
  아니라 "검정력 부족"일 수 있고, 그건 결과 카드에 그대로 적는다.
- 실험4의 분류기는 **2 seed·QUICK**으로 학습된 것이다. 조건부 규칙은 그 분류기의
  오답 구조에 맞춰진 것이므로, 분류기를 바꾸면 계수도 다시 적합해야 한다.
- 참 클래스 오라클도 **A조에서 적합**한다. 진짜 상한(테스트셋에서 직접 적합)보다는
  낮게 나오지만, 그게 공정한 비교다.
